# Assignment No. 2: Data Cleaning and Preprocessing

## 1. Introduction

Data cleaning and preprocessing are important steps in data analysis and machine learning. Raw datasets may contain missing values, duplicate records, inconsistent categories, incorrect data types, and unusual numerical observations.

The purpose of this assignment is to identify these problems and apply suitable methods to prepare a dataset for further analysis and machine learning. Each cleaning and preprocessing decision will be explained and supported by code and outputs.


In [1]:
from google.colab import files

uploaded = files.upload()

Saving StudentPerformanceFactors.csv to StudentPerformanceFactors.csv


## 2. Dataset Description and Loading

**Dataset Title:** Student Performance Factors

**Source:** Kaggle — Student Performance Factors dataset.

**Objective:** This dataset contains information about students' study habits, attendance, sleep, previous scores, tutoring, and other factors that may be associated with academic performance. The target variable for this assignment is Exam_Score.

The dataset is loaded into a Pandas DataFrame for inspection and cleaning. The actual number of rows and columns will be reported from the uploaded file.


In [2]:
import pandas as pd
import numpy as np

# Find the uploaded CSV filename
csv_files = [name for name in uploaded.keys()
             if name.lower().endswith(".csv")]

if not csv_files:
    raise ValueError("Please upload a CSV dataset file.")

file_name = csv_files[0]

# Load the dataset
df = pd.read_csv(file_name)

# Keep an original copy
df_raw = df.copy()

print("Dataset loaded successfully!")
print("Dataset title: Student Performance Factors")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Dataset title: Student Performance Factors
Number of rows: 6607
Number of columns: 20


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


## 3. Initial Data Inspection

The dataset is inspected to understand its structure, column names, data types, missing values, and statistical characteristics. This initial inspection helps identify potential data quality problems before applying cleaning methods.


In [3]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

Dataset shape: (6607, 20)

Column names:
['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender', 'Exam_Score']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Hours_Studied,6607.0,NaN,NaN,NaN,19.975329,5.990594,1.0,16.0,20.0,24.0,44.0
Attendance,6607.0,NaN,NaN,NaN,79.977448,11.547475,60.0,70.0,80.0,90.0,100.0
Parental_Involvement,6607,3,Medium,3362,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Access_to_Resources,6607,3,Medium,3319,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Extracurricular_Activities,6607,2,Yes,3938,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sleep_Hours,6607.0,NaN,NaN,NaN,7.02906,1.46812,4.0,6.0,7.0,8.0,10.0
Previous_Scores,6607.0,NaN,NaN,NaN,75.070531,14.399784,50.0,63.0,75.0,88.0,100.0
Motivation_Level,6607,3,Medium,3351,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Internet_Access,6607,2,Yes,6108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tutoring_Sessions,6607.0,NaN,NaN,NaN,1.493719,1.23057,0.0,1.0,1.0,2.0,8.0


**Interpretation:** The `info()` function displays the data types and non-null counts of each column, while the statistical summary shows the basic statistical characteristics of the dataset.


## 4. Identifying Missing Values

Missing values occur when information is unavailable in a dataset. They are identified by calculating the number and percentage of missing entries in each column. The results are used to select an appropriate treatment method.


In [4]:
missing_table = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

display(missing_table.sort_values(
    by="Missing Count", ascending=False
))

,Missing Count,Missing Percentage
Parental_Education_Level,90,1.36
Teacher_Quality,78,1.18
Distance_from_Home,67,1.01
Hours_Studied,0,0.00
Access_to_Resources,0,0.00
Parental_Involvement,0,0.00
Attendance,0,0.00
Extracurricular_Activities,0,0.00
Motivation_Level,0,0.00
Internet_Access,0,0.00


**Interpretation:** This output shows the number and percentage of missing values in each column. If all values are zero, it means that no missing values were detected in the dataset.


## 5. Detecting and Removing Duplicate Records

Duplicate rows may cause observations to be counted more than once. Exact duplicate records are counted and removed. The dataset size is checked again to verify the result.


In [5]:
duplicates_before = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)

df = df.drop_duplicates().copy()

print("Duplicate rows after removal:", df.duplicated().sum())
print("Shape after duplicate removal:", df.shape)

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Shape after duplicate removal: (6607, 20)


**Interpretation:** If no duplicate rows are found, no rows will be removed. If duplicate rows exist, the total number of rows will decrease after removing them.


## 6. Correcting Inconsistent Values

Extra spaces and inconsistent capitalization can create multiple categories for the same value. Column names and text values are standardized to improve consistency. Categories are inspected before making any further corrections.


In [6]:
# Clean column names
df.columns = df.columns.str.strip()

# Remove extra spaces and standardize text capitalization
for col in df.select_dtypes(include=["object", "category"]).columns:
    df[col] = df[col].apply(
        lambda x: x.strip().title() if isinstance(x, str) else x
    )

# Inspect unique categories
for col in df.select_dtypes(include=["object", "category"]).columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


Parental_Involvement:
Parental_Involvement
Medium    3362
High      1908
Low       1337
Name: count, dtype: int64

Access_to_Resources:
Access_to_Resources
Medium    3319
High      1975
Low       1313
Name: count, dtype: int64

Extracurricular_Activities:
Extracurricular_Activities
Yes    3938
No     2669
Name: count, dtype: int64

Motivation_Level:
Motivation_Level
Medium    3351
Low       1937
High      1319
Name: count, dtype: int64

Internet_Access:
Internet_Access
Yes    6108
No      499
Name: count, dtype: int64

Family_Income:
Family_Income
Low       2672
Medium    2666
High      1269
Name: count, dtype: int64

Teacher_Quality:
Teacher_Quality
Medium    3925
High      1947
Low        657
NaN         78
Name: count, dtype: int64

School_Type:
School_Type
Public     4598
Private    2009
Name: count, dtype: int64

Peer_Influence:
Peer_Influence
Positive    2638
Neutral     2592
Negative    1377
Name: count, dtype: int64

Learning_Disabilities:
Learning_Disabilities
No     5912
Yes

Interpretation: Now, check the unique values in each categorical column. If categories with the same meaning have different spellings, correct them. Standardizing capitalization can make values such as yes and Yes consistent

## 7. Checking and Correcting Data Types

Data types determine how values are stored and processed. Numerical columns should contain numbers, while categorical columns should represent labels or groups. Data types are inspected and corrected only when necessary.


In [7]:
print("Data types before correction:")
print(df.dtypes)

# Convert numeric-looking text columns only when values are valid
for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")

        # Convert only if all originally non-missing values are numeric
        if df[col].notna().sum() == converted.notna().sum():
            df[col] = converted

print("\nData types after inspection:")
print(df.dtypes)

Data types before correction:
Hours_Studied                  int64
Attendance                     int64
Parental_Involvement          object
Access_to_Resources           object
Extracurricular_Activities    object
Sleep_Hours                    int64
Previous_Scores                int64
Motivation_Level              object
Internet_Access               object
Tutoring_Sessions              int64
Family_Income                 object
Teacher_Quality               object
School_Type                   object
Peer_Influence                object
Physical_Activity              int64
Learning_Disabilities         object
Parental_Education_Level      object
Distance_from_Home            object
Gender                        object
Exam_Score                     int64
dtype: object

Data types after inspection:
Hours_Studied                  int64
Attendance                     int64
Parental_Involvement          object
Access_to_Resources           object
Extracurricular_Activities    object
S

Interpretation: Columns that already have the correct data types do not need to be changed. Also, inspect the columns for date/time variables. If the dataset does not contain any date/time columns, clearly mention this in the report

## 8. Outlier Detection Using the IQR Method

Outliers are observations that fall unusually far from the central distribution of a numerical variable. The Interquartile Range (IQR) method identifies potential outliers using the first and third quartiles.

Potential outliers are investigated rather than automatically deleted because unusual values may be valid observations.


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_results = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    outlier_results.append({
        "Column": col,
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2),
        "Potential Outliers": int(count)
    })

outlier_table = pd.DataFrame(outlier_results)
display(outlier_table)

,Column,Lower Bound,Upper Bound,Potential Outliers
0,Hours_Studied,4.0,36.0,43
1,Attendance,40.0,120.0,0
2,Sleep_Hours,3.0,11.0,0
3,Previous_Scores,25.5,125.5,0
4,Tutoring_Sessions,-0.5,3.5,430
5,Physical_Activity,-1.0,7.0,0
6,Exam_Score,59.0,75.0,104


Interpretation: The table will show potential outliers in each numerical column. Being identified as an outlier by the IQR method does not automatically mean that the data is incorrect.
We will not delete valid student records without a valid reason. If any impossible or clearly incorrect values are found, we will investigate them and apply an appropriate treatment with proper justification.

## 9. Identifying Features and Target Variable

Features are the input variables used to predict or explain an outcome. The target variable is the outcome that the model aims to predict.

For this assignment, Exam_Score is selected as the target variable, while the remaining appropriate columns are considered input features. The target is excluded from feature preprocessing to prevent data leakage.


In [9]:
target_col = "Exam_Score"

if target_col not in df.columns:
    raise ValueError(
        "Exam_Score was not found. Check the dataset column names."
    )

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:", target_col)
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Feature columns:
['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']

Target column: Exam_Score
Features shape: (6607, 19)
Target shape: (6607,)


Interpretation: X represents the input features, while y represents the target variable. The exam score should not be included in the input features because it is the variable we want to predict.

## 10. Classification of Variables

Variables are classified according to their meaning and data type. Numerical variables contain measurable quantities, categorical variables represent groups, binary variables have two categories, and ordinal variables have categories with a meaningful order. Date/time variables represent dates or times, if present.

This classification helps select appropriate encoding and scaling methods.


In [10]:
# Numerical and categorical variables
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Binary categorical variables
binary_cols = [
    col for col in categorical_cols
    if X[col].nunique(dropna=True) == 2
]

# Ordinal columns identified by their meaning
possible_ordinal = [
    "Parental_Involvement",
    "Access_to_Resources",
    "Motivation_Level",
    "Family_Income",
    "Teacher_Quality",
    "Peer_Influence",
    "Parental_Education_Level",
    "Distance_from_Home"
]

ordinal_cols = [col for col in possible_ordinal if col in X.columns]

# Date/time columns
date_cols = [
    col for col in X.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("Numerical variables:", numerical_cols)
print("Categorical variables:", categorical_cols)
print("Binary categorical variables:", binary_cols)
print("Potential ordinal variables:", ordinal_cols)
print("Date/time variables:", date_cols)

Numerical variables: ['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']
Categorical variables: ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']
Binary categorical variables: ['Extracurricular_Activities', 'Internet_Access', 'School_Type', 'Learning_Disabilities', 'Gender']
Potential ordinal variables: ['Parental_Involvement', 'Access_to_Resources', 'Motivation_Level', 'Family_Income', 'Teacher_Quality', 'Peer_Influence', 'Parental_Education_Level', 'Distance_from_Home']
Date/time variables: []


Important: Verify the list of ordinal columns against the actual unique values in the dataset. If the dataset does not contain any date/time columns, it is perfectly fine for the output to show Date/time variables: [].

## 11. Feature Engineering

Feature engineering creates useful input variables from existing data. A study-attendance interaction feature is created by combining hours studied and attendance. This feature may help represent the combined relationship between study time and attendance.

The feature is created using input variables only, without using the target variable.


In [11]:
if "Hours_Studied" in X.columns and "Attendance" in X.columns:
    X["Study_Attendance_Interaction"] = (
        X["Hours_Studied"] * X["Attendance"] / 100
    )
    print("Study_Attendance_Interaction created.")
else:
    print("Required columns were not found; no interaction feature created.")

print("Updated feature shape:", X.shape)
display(X.head())

Study_Attendance_Interaction created.
Updated feature shape: (6607, 20)


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Study_Attendance_Interaction
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,19.32
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,12.16
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,23.52
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,25.81
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,17.48


Interpretation: If both required columns are present, the new feature will be created successfully. This is an optional derived feature, and its usefulness can be verified during model evaluation.

## 12. Splitting Data into Training and Testing Sets

The dataset is divided into training and testing sets before fitting preprocessing methods. The training set is used to learn imputation and scaling parameters, while the testing set is reserved for evaluation.

This procedure helps prevent data leakage and gives a more reliable assessment of model performance.


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (5285, 20)
Testing features: (1322, 20)
Training target: (5285,)
Testing target: (1322,)


Interpretation: 80% of the data will be used for training, and 20% will be used for testing. Setting random_state=42 ensures that the data split is reproducible, producing the same split each time the code is run.

## 13. Checking Skewness and Transforming Numerical Features

Skewness measures the asymmetry of a numerical distribution. Strongly right-skewed, non-negative features may benefit from a log1p transformation. Skewness is checked using the training data, and any selected transformation is applied consistently to both training and testing sets.


In [13]:
# Check skewness using training data only
train_numeric = X_train.select_dtypes(include=np.number)

skew_values = train_numeric.skew().sort_values(
    key=lambda s: s.abs(),
    ascending=False
)

print("Training feature skewness:")
display(skew_values.to_frame("Skewness"))

# Apply log1p to Tutoring_Sessions only if strongly right-skewed
if "Tutoring_Sessions" in X_train.columns:
    skew = X_train["Tutoring_Sessions"].skew()

    if skew > 1 and (X_train["Tutoring_Sessions"].dropna() >= 0).all():
        X_train["Tutoring_Sessions"] = np.log1p(
            X_train["Tutoring_Sessions"]
        )
        X_test["Tutoring_Sessions"] = np.log1p(
            X_test["Tutoring_Sessions"]
        )
        print("Applied log1p transformation to Tutoring_Sessions.")
    else:
        print("Log transformation was not required for Tutoring_Sessions.")
else:
    print("Tutoring_Sessions column not found.")

Training feature skewness:


,Skewness
Tutoring_Sessions,0.827026
Study_Attendance_Interaction,0.289446
Sleep_Hours,-0.034304
Physical_Activity,-0.031783
Attendance,0.025000
Previous_Scores,-0.001134
Hours_Studied,-0.000383


Log transformation was not required for Tutoring_Sessions.


Interpretation: We apply a log transformation only if Tutoring_Sessions is strongly right-skewed in the training data and contains non-negative values. If these conditions are not met, we should not force the transformation.

## 14. Encoding, Imputation, and Standardization

Numerical missing values are imputed using the median, while categorical missing values are imputed using the most frequent category. Nominal categorical variables are encoded using one-hot encoding, while ordinal variables are encoded according to their meaningful order.

Numerical input features are standardized using StandardScaler. All preprocessing methods are fitted on the training data only and then applied to the testing data. The target variable is not scaled or encoded as an input feature.


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

# Identify column types after feature engineering
num_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

cat_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Define meaningful order for known ordinal variables
ordinal_orders = {
    "Parental_Involvement": ["Low", "Medium", "High"],
    "Access_to_Resources": ["Low", "Medium", "High"],
    "Motivation_Level": ["Low", "Medium", "High"],
    "Family_Income": ["Low", "Medium", "High"],
    "Teacher_Quality": ["Low", "Medium", "High"],
    "Peer_Influence": ["Negative", "Neutral", "Positive"],
    "Parental_Education_Level": [
        "High School", "College", "Postgraduate"
    ],
    "Distance_from_Home": ["Near", "Moderate", "Far"]
}

# Use ordinal encoding only for columns actually present
ordinal_features = [
    col for col in cat_features
    if col in ordinal_orders
]

# Remaining categorical columns use one-hot encoding
nominal_features = [
    col for col in cat_features
    if col not in ordinal_features
]

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Nominal categorical preprocessing
nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Ordinal categorical preprocessing
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[
            ordinal_orders[col] for col in ordinal_features
        ],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("numerical", numeric_pipeline, num_features),
    ("nominal", nominal_pipeline, nominal_features),
    ("ordinal", ordinal_pipeline, ordinal_features)
])

# Fit only on training data; transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully!")
print("Training data after preprocessing:", X_train_processed.shape)
print("Testing data after preprocessing:", X_test_processed.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Preprocessing completed successfully!
Training data after preprocessing: (5285, 25)
Testing data after preprocessing: (1322, 25)
Training target shape: (5285,)
Testing target shape: (1322,)


**Interpretation:** The preprocessing pipeline was executed successfully. The training dataset contains 5,285 rows and 25 processed features, while the testing dataset contains 1,322 rows and the same 25 features. The target variable is kept separate from the input features. The increase in feature columns may be due to categorical encoding.


## 15. Final Preprocessing Verification

The final training and testing datasets are checked to confirm that preprocessing has completed successfully. Their dimensions are compared, and the transformed data is checked for invalid numerical values. The target variable remains separate from the processed input features.


In [15]:
from scipy import sparse

def check_finite(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return np.isfinite(values).all()

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("Training features contain only finite values:",
      check_finite(X_train_processed))

print("Testing features contain only finite values:",
      check_finite(X_test_processed))

print("Missing values in training target:", y_train.isna().sum())
print("Missing values in testing target:", y_test.isna().sum())

print("\nFinal verification completed.")

Training shape: (5285, 25)
Testing shape: (1322, 25)
Training features contain only finite values: True
Testing features contain only finite values: True
Missing values in training target: 0
Missing values in testing target: 0

Final verification completed.


Interpretation: Both processed feature sets should contain only finite values. If any missing values are found in the target variable, those rows must be handled appropriately before training any model.

## 16. Summary of Data Cleaning and Preprocessing

The dataset was inspected for missing values, duplicate records, inconsistent text values, incorrect data types, and potential outliers. Appropriate cleaning decisions were made based on the inspection results.

The input features were classified, and suitable methods were selected for imputation, categorical encoding, numerical standardization, and skewness transformation where required. A derived feature was created where the required columns were available.

The data was split into training and testing sets before fitting the preprocessing pipeline. This helped prevent data leakage and ensured that preprocessing parameters were learned from the training data only.


In [16]:
print("FINAL ASSIGNMENT SUMMARY")
print("-" * 40)

print("Original dataset shape:", df_raw.shape)
print("Cleaned dataset shape:", df.shape)

print("Duplicate rows remaining:", df.duplicated().sum())
print("Missing values remaining in cleaned raw data:",
      df.isnull().sum().sum())

print("Training samples:", X_train_processed.shape[0])
print("Testing samples:", X_test_processed.shape[0])
print("Processed training features:", X_train_processed.shape[1])
print("Processed testing features:", X_test_processed.shape[1])

print("Target variable:", target_col)
print("Final verification:", check_finite(X_train_processed)
      and check_finite(X_test_processed))

FINAL ASSIGNMENT SUMMARY
----------------------------------------
Original dataset shape: (6607, 20)
Cleaned dataset shape: (6607, 20)
Duplicate rows remaining: 0
Missing values remaining in cleaned raw data: 235
Training samples: 5285
Testing samples: 1322
Processed training features: 25
Processed testing features: 25
Target variable: Exam_Score
Final verification: True


Important: Missing values may still remain in the raw cleaned data because they are handled by the training-only preprocessing pipeline. This does not mean that the preprocessing process has failed.

https://www.kaggle.com/datasets/lainguyn123/student-performance-factors